# Task 2d: Delta / Spindle Coupling

Delta-spindle coupling on the same seven rats as the rest of Task 2.

### Pairings

All nine delta-area x spindle-area pairs (PL/RSC/HPC x PL/RSC/HPC), Yixiao Oct 2026.

### Window

Delta -> Spindle, 100-1300 ms peak-to-peak (`PROJECT_DATA_REFERENCE.md`).

`lag = spindle_peak - delta_peak`, positive = spindle after delta.

PL-PL, RSC-RSC and HPC-HPC use the same electrode for both waves (possible leakage), the six cross-area pairs don't.

---
# Load and restrict

## Configuration

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# lab repo root on the path, so `modules` imports work from this folder
REPO_ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "modules" / "project_config.py").exists()
)
sys.path.insert(0, str(REPO_ROOT))
from modules.project_config import get_path  # noqa: E402

# local paths come from config.yaml (see config.example.yaml)
ROOT = get_path("R1_8_root")  # holds Rat_HM_Ephys_TD_Analysis_R1_8 and _R9_16
assert get_path("R9_16_root") == ROOT, "these notebooks expect R1_8_root == R9_16_root"
WAVE_CACHE = get_path("event_cache_root")
SESSION_MAP_CSV = get_path("session_map_csv")
DAY_GRID_XLSX = get_path("day_grid_xlsx")
OUT = REPO_ROOT / "results" / "leticia_event_coupling"  # figures and caches
SPINDLE_METHOD = "wavelet_amp_1_ampcore_3"
PHASES = ["postsleep"]

BOUT_KEYS = ["rat", "date", "trial", "nrem_bout_index"]

# all nine pairings (Yixiao, Oct 2026)
AREAS = ["PL", "RSC", "HPC"]
PAIRS = [(d, s) for d in AREAS for s in AREAS]

WINDOW_MS = (100, 1300)  # Delta -> Spindle, peak-to-peak (PROJECT_DATA_REFERENCE.md)

# rats with trusted ripples (Yixiao, 17 Aug 2026), kept for consistency
RATS = [1, 3, 7, 8, 11, 12, 13]

FIGDIR = OUT / "task2d_figures"
FIGDIR.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name):
    """Save a figure. Takes the figure object explicitly -- plt.gcf() is
    unreliable after tight_layout() and returns a blank figure after show()."""
    fig.savefig(FIGDIR / f"{name}.png", dpi=200, bbox_inches="tight")
    print("saved:", name)


print("drive reachable:", ROOT.exists())
print("rats in analysis:", RATS)
print("pairs:", PAIRS, "| window:", WINDOW_MS)

## Load and restrict

Loads from the event cache (`event_cache_root` in `config.yaml`); `add_condition` adds HC / Training.

In [ ]:
from modules.event_coupling.build_tables import add_condition, build_wave

deltas = add_condition(
    build_wave(ROOT, "delta", SPINDLE_METHOD, PHASES, WAVE_CACHE, rebuild=False),
    SESSION_MAP_CSV,
)
spindles = add_condition(
    build_wave(ROOT, "spindle", SPINDLE_METHOD, PHASES, WAVE_CACHE, rebuild=False),
    SESSION_MAP_CSV,
)

print(f"all 16 rats : {len(deltas):>10,} deltas | {len(spindles):>10,} spindles")

# restrict to the seven, and drop Maze_Hab / Novelty / aborted days
dlt = deltas[(deltas["rat"].isin(RATS)) & (deltas["group"] != "other")]
spn = spindles[
    (spindles["rat"].isin(RATS))
    & (spindles["group"] != "other")
    & (spindles["area"].isin(["PL", "RSC", "HPC"]))
]

print(f"seven rats  : {len(dlt):>10,} deltas | {len(spn):>10,} spindles")
print(
    f"             ({100 * len(dlt) / len(deltas):.0f}% of deltas, "
    f"{100 * len(spn) / len(spindles):.0f}% of spindles)"
)
print(dlt.groupby("group").size().to_string())

## Goal locations, session types and HC context

- `gl_num` / `session_num` from the StudyDay workbooks (`gl_sessions.py`)
- `session_type`: S1 = new goal location, S2/S4 = probe, S3/S5 = normal
- HC context: goal location before, next session, retention delay

In [ ]:
from modules.event_coupling.gl_sessions import add_gl_sessions, build_gl_sessions
from modules.event_coupling.hc_sessions import (
    add_session_type,
    attach_hc_context,
    build_day_table,
)


def prep_keys(df):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    df["rat"] = df["rat"].astype("int64")
    return df


gl = prep_keys(build_gl_sessions(ROOT, cache_dir=WAVE_CACHE, rebuild=False))
days = add_session_type(build_day_table(DAY_GRID_XLSX, verbose=False))
hc_ctx = attach_hc_context(days, gl_table=gl, verbose=False)

SESSION_COLS = ["rat", "date", "session_type"]
HC_COLS = [
    "rat",
    "date",
    "hc_phase",
    "follows_gl",
    "next_type",
    "retention_delay",
    "is_consolidation",
]


def add_context(df):
    df = add_gl_sessions(prep_keys(df), gl)
    df = df.merge(
        prep_keys(days)[SESSION_COLS].drop_duplicates(), on=["rat", "date"], how="left"
    )
    df = df.merge(
        prep_keys(hc_ctx)[HC_COLS].drop_duplicates(), on=["rat", "date"], how="left"
    )
    return df


dlt_ctx = add_context(dlt)
spn_ctx = add_context(spn)

print(dlt_ctx.groupby(["group", "session_type"], dropna=False).size().to_string())

### Enough data in each group?

With seven rats a sign-flip test can't reach p < 0.05 below n = 4.

In [ ]:
def cell_rats(df, keys, label):
    d = df.dropna(subset=[k for k in keys if k != "group"])
    g = (
        d.groupby(keys, dropna=False, observed=True)
        .agg(rats=("rat", "nunique"), bouts=("nrem_bout_index", "size"))
        .reset_index()
    )
    g["n_ok"] = np.where(
        g["rats"] >= 6, "yes", np.where(g["rats"] >= 4, "marginal", "NO")
    )
    print(f"--- {label} ---")
    print(g.to_string(index=False))
    print()


cell_rats(
    dlt_ctx[dlt_ctx["group"] == "HC"], ["retention_delay"], "HC by retention delay"
)
cell_rats(
    dlt_ctx[dlt_ctx["group"] == "Training"],
    ["session_type"],
    "Training by session type",
)
cell_rats(dlt_ctx, ["cohort", "group"], "cohort x group")

---
# Fixed-window count

Per pair, since spindles have their own area.

## Fixed-window count

Spindle peak 100-1300 ms after a delta peak.

In [ ]:
from modules.event_coupling.coupling import count_sequences

ds = {}
for d_area, s_area in PAIRS:
    pair_label = f"{d_area} delta - {s_area} spindle"
    r = count_sequences(
        dlt[dlt["area"] == d_area],
        spn,
        "delta",
        "spindle",
        window_ms=WINDOW_MS,
        second_area=s_area,
    )
    ds[(d_area, s_area)] = r
    print(
        f"{pair_label:<22}: bouts {len(r):>7,} | total D->S sequences "
        f"{r['n_sequences'].sum():>9,} | spindles in those bouts "
        f"{r['n_spindle'].sum():>9,}"
    )

### Yixiao's counting rule

Each spindle counted once. `window_ms` passed directly (no delta-spindle entry in `WINDOWS_MS`).

Analytic chance `1 - exp(-r*w)` overestimates the effect (delta waves are rhythmic), the surrogate below is the real null.

In [ ]:
from modules.event_coupling.coupling_unique import count_sequences_unique

for d_area, s_area in PAIRS:
    pair_label = f"{d_area} delta - {s_area} spindle"
    u = count_sequences_unique(
        dlt[dlt["area"] == d_area],
        spn[spn["area"] == s_area],
        "delta",
        "spindle",
        window_ms=WINDOW_MS,
    )
    obs, n = u["n_sequences"].sum(), u["n_spindle"].sum()
    print(
        f"{pair_label:<22}: {obs:>8,} of {n:>8,} spindles coupled ({100 * obs / n:.1f}%)"
    )

## Bout durations

Bouts have to be longer than the shift used in the negative control.

In [ ]:
bout_dur = dlt[BOUT_KEYS + ["nrem_bout_duration_s"]].drop_duplicates()
print(bout_dur["nrem_bout_duration_s"].describe().round(1).to_string())

## Negative control with a shifted window

Same 1200 ms width, starting 5000 ms after each delta.

| ratio (control / real) | meaning |
|---|---|
| well below 1 | good, we're measuring timing |
| near 1 | bad, we're measuring event density |

In [ ]:
WINDOW_WIDTH_MS = WINDOW_MS[1] - WINDOW_MS[0]  # 1200 ms
CTRL_WINDOW_MS = (WINDOW_MS[0] + 5000, WINDOW_MS[1] + 5000)  # (5100, 6300)

for d_area, s_area in PAIRS:
    pair_label = f"{d_area} delta - {s_area} spindle"
    ctrl = count_sequences(
        dlt[dlt["area"] == d_area],
        spn,
        "delta",
        "spindle",
        window_ms=CTRL_WINDOW_MS,
        second_area=s_area,
    )
    real_total, ctrl_total = (
        ds[(d_area, s_area)]["n_sequences"].sum(),
        ctrl["n_sequences"].sum(),
    )
    print(
        f"{pair_label:<22}: real {real_total:>9,} | control {ctrl_total:>9,} "
        f"| ratio {ctrl_total / real_total:.3f}"
    )

## What chance alone predicts

Pairs expected from event density alone (deltas x spindle rate x window width).

In [ ]:
total_nrem_s = bout_dur["nrem_bout_duration_s"].sum()
window_width_s = WINDOW_WIDTH_MS / 1000.0

print(f"total NREM time      : {total_nrem_s:>12,.0f} s")
print(f"window width         : {window_width_s:>12.3f} s")
for d_area, s_area in PAIRS:
    pair_label = f"{d_area} delta - {s_area} spindle"
    n_delta = (dlt["area"] == d_area).sum()
    n_spn_area = (spn["area"] == s_area).sum()
    spindle_rate = n_spn_area / total_nrem_s
    expected = spindle_rate * window_width_s * n_delta
    observed = ds[(d_area, s_area)]["n_sequences"].sum()
    print(
        f"{pair_label:<22}: expected {expected:>10,.0f} | observed {observed:>10,} "
        f"| observed/expected {observed / expected:>7.3f}"
    )

---
# Lag histogram

Peri-event histograms, peak latencies, compared with a surrogate (slide 13).

Wider edges and baseline than delta-ripple (window is 100-1300 ms).

In [ ]:
from modules.event_coupling import lag_histogram as lh

MAX_LAG_MS = 1600
EDGES = np.arange(-1600, 1620, 20)
BASELINE_MS = 1400  # tail = |lag| >= 1400, clear of the 100-1300 ms window
left = EDGES[:-1]

WINDOWS = [("D->S", WINDOW_MS[0], WINDOW_MS[1])]

### Check before using

Self-test with a planted lag. If it prints FAIL, stop.

In [ ]:
lh.self_test(planted_lag_ms=200)

### Working sample

2,000 bouts for the surrogate (the histograms use all bouts).

In [ ]:
all_bouts = pd.concat(
    [r[BOUT_KEYS] for r in ds.values()], ignore_index=True
).drop_duplicates()
sample = all_bouts.sample(n=min(2000, len(all_bouts)), random_state=0)

dlt_s = dlt.merge(sample, on=BOUT_KEYS)
spn_s = spn.merge(sample, on=BOUT_KEYS)

print("sample bouts:", len(sample))
print("sample rows :", f"{len(dlt_s):,} deltas |", f"{len(spn_s):,} spindles")

## Histogram per pair, full dataset

Per pair. Baseline from the tails (|lag| >= 1400 ms).

y = raw pair counts, so compare shape and peak position between pairs, not height.

In [ ]:
res_full = {}
for d_area, s_area in PAIRS:
    pair_label = f"{d_area} delta - {s_area} spindle"
    lg = lh.collect_lags(
        dlt[dlt["area"] == d_area],
        spn[spn["area"] == s_area],
        "delta",
        "spindle",
        max_lag_ms=MAX_LAG_MS,
    )
    res_full[(d_area, s_area)] = lh.summarise(
        lg, EDGES, baseline_ms=BASELINE_MS, windows=WINDOWS, label=pair_label
    )
    del lg

for key, r in res_full.items():
    print(
        f"{key}: baseline/bin {r['baseline']:.1f} | noise floor ~{lh.noise_floor_pct(r):.1f}% of baseline"
    )

In [ ]:
for (d_area, s_area), r in res_full.items():
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(left, r["counts"], color="tab:blue", lw=1.6)
    ax.axhline(r["baseline"], color="k", lw=1, ls=":", label="baseline")
    ax.axvspan(
        WINDOW_MS[0], WINDOW_MS[1], color="tab:blue", alpha=0.10, label="D->S window"
    )
    ax.axvline(0, color="k", lw=1)
    ax.set_xlabel("spindle peak - delta peak (ms)   [positive = spindle after delta]")
    ax.set_ylabel("count")
    ax.set_title(f"{d_area} delta vs {s_area} spindle -- full dataset")
    ax.legend(fontsize=8)
    plt.tight_layout()
    save_fig(fig, f"02_histogram_full_{d_area}_{s_area}")
    plt.show()

## End-of-delta -> start-of-spindle

Separates delta waves that end before the spindle starts from overlapping pairs. Uses `delta_end_index` -> `spindle_start_index`.

`lag = spindle_start - delta_end`: lag < 0 overlap, lag >= 0 sequential.

In [ ]:
# wider range and baseline than the peak-to-peak histograms
MAX_LAG_MS_ES = 1600
EDGES_ES = np.arange(-1600, 1620, 20)
BASELINE_ES = 1400
left_es = EDGES_ES[:-1]

res_end_start = {}
for d_area, s_area in PAIRS:
    pair_label = f"{d_area} delta - {s_area} spindle"
    lg = lh.collect_lags_cols(
        dlt[dlt["area"] == d_area],
        spn[spn["area"] == s_area],
        "delta_end_index",
        "spindle_start_index",
        max_lag_ms=MAX_LAG_MS_ES,
    )
    res_end_start[(d_area, s_area)] = lh.summarise(
        lg,
        EDGES_ES,
        baseline_ms=BASELINE_ES,
        windows=[
            ("overlap (lag<0)", -MAX_LAG_MS_ES, 0),
            ("sequential (lag>=0)", 0, MAX_LAG_MS_ES),
        ],
        label=pair_label,
    )

for key, r in res_end_start.items():
    ov = r["windows"]["overlap (lag<0)"]["observed"]
    sq = r["windows"]["sequential (lag>=0)"]["observed"]
    total = ov + sq
    pct = 100 * sq / total if total else float("nan")
    print(
        f"{key}: {ov:,} pairs overlap (lag<0) | {sq:,} pairs sequential (lag>=0) "
        f"| {pct:.1f}% sequential"
    )

In [ ]:
for (d_area, s_area), r in res_end_start.items():
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(left_es, r["counts"], color="tab:purple", lw=1.6)
    ax.axhline(r["baseline"], color="k", lw=1, ls=":", label="baseline")
    ax.axvline(0, color="k", lw=1.2)
    ax.axvspan(
        EDGES_ES[0], 0, color="tab:red", alpha=0.06, label="overlap (already counted)"
    )
    ax.axvspan(0, EDGES_ES[-1], color="tab:green", alpha=0.06, label="sequential (new)")
    ax.set_xlabel(
        "spindle start - delta end (ms)   [<0 = overlapping, >=0 = sequential]"
    )
    ax.set_ylabel("count")
    ax.set_title(f"{d_area} delta end -> {s_area} spindle start -- full dataset")
    ax.legend(fontsize=8)
    plt.tight_layout()
    save_fig(fig, f"15_histogram_end_start_{d_area}_{s_area}")
    plt.show()

---
# Surrogate null

## Can every bout be shifted?

Bouts shorter than twice the minimum shift are skipped. Fine if under ~1% of events are lost.

In [ ]:
dur = bout_dur["nrem_bout_duration_s"]
short = bout_dur.loc[dur <= 2.0, BOUT_KEYS]
lost_d = dlt.merge(short, on=BOUT_KEYS).shape[0]
lost_s = spn.merge(short, on=BOUT_KEYS).shape[0]

print(
    f"bouts too short at the 1 s default: {(dur <= 2).sum()} of {len(dur)} "
    f"({100 * (dur <= 2).mean():.2f}%)"
)
print(
    f"  deltas lost   {lost_d:>8,} of {len(dlt):>9,} ({100 * lost_d / len(dlt):.3f}%)"
)
print(
    f"  spindles lost {lost_s:>8,} of {len(spn):>9,} ({100 * lost_s / len(spn):.3f}%)"
)

## Circular-shift surrogate

Spindle train shifted within each recording's NREM time.

1,000 iterations (Yixiao, Oct 2026) on the 2,000-bout working sample.

Tail z should be close to zero.

In [ ]:
import pickle

from modules.event_coupling import surrogate as sg

CKPT_DIR = OUT / "task2d_checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)


def make_checkpoint_fn(tag):
    path = CKPT_DIR / f"{tag}.pkl"

    def _fn(partial):
        with open(path, "wb") as f:
            pickle.dump(partial, f)
        print(f"    checkpoint saved: {path.name} ({partial['n_done']}/1000 iters)")

    return _fn


res_surr = {}
for d_area, s_area in PAIRS:
    pair_label = f"{d_area} delta - {s_area} spindle"
    print(f"--- {pair_label} ---")
    ckpt_path = CKPT_DIR / f"step13_{d_area}_{s_area}.pkl"
    resume = None
    if ckpt_path.exists():
        with open(ckpt_path, "rb") as f:
            resume = pickle.load(f)

    if resume is not None and resume["n_done"] >= 1000:
        print(
            f"    already complete at {resume['n_done']}/1000 -- using checkpoint directly"
        )
        res_surr[(d_area, s_area)] = resume
        continue

    if resume is not None:
        print(f"    resuming from {resume['n_done']}/1000 iterations")
    res_surr[(d_area, s_area)] = sg.surrogate_distribution(
        dlt_s[dlt_s["area"] == d_area],
        spn_s[spn_s["area"] == s_area],
        "delta",
        "spindle",
        EDGES,
        n_iter=1000,
        seed=0,
        max_lag_ms=MAX_LAG_MS,
        scope="recording",
        checkpoint_every=100,
        checkpoint_fn=make_checkpoint_fn(f"step13_{d_area}_{s_area}"),
        resume_from=resume,
    )

In [ ]:
win = (left >= WINDOW_MS[0]) & (left < WINDOW_MS[1])
tail = (left < -BASELINE_MS) | (left >= BASELINE_MS)

print(f"{'pair':<24}{'peak z':>9}{'at ms':>8}{'window z':>11}{'tail z':>9}")
for (d_area, s_area), r in res_surr.items():
    pair_label = f"{d_area} delta - {s_area} spindle"
    print(
        f"{pair_label:<24}{np.nanmax(r['z']):>9.1f}{int(left[np.nanargmax(r['z'])]):>8}"
        f"{np.nanmean(r['z'][win]):>11.1f}{np.nanmean(r['z'][tail]):>9.2f}"
    )

In [ ]:
# 3x3 grid: rows = delta area, columns = spindle area (AREAS order).
fig, axes = plt.subplots(3, 3, figsize=(14, 11), sharex=True, sharey=True)
for (d_area, s_area), r in res_surr.items():
    ax = axes[AREAS.index(d_area), AREAS.index(s_area)]
    ax.fill_between(
        left,
        r["mu"] - 2 * r["sd"],
        r["mu"] + 2 * r["sd"],
        color="0.75",
        alpha=0.6,
        label="surrogate +/-2 SD",
    )
    ax.plot(left, r["mu"], color="0.4", lw=1, label="surrogate mean")
    ax.plot(left, r["real"], color="tab:red", lw=1.6, label="observed")
    ax.axvline(0, color="k", lw=0.8)
    ax.set_title(f"{d_area} delta vs {s_area} spindle", fontsize=10)
for i, d_area in enumerate(AREAS):
    axes[i, 0].set_ylabel(f"count\n({d_area} delta)")
for j, s_area in enumerate(AREAS):
    axes[-1, j].set_xlabel(f"lag (ms)\n({s_area} spindle)")
axes[0, 0].legend(fontsize=8)
fig.suptitle("Circular-shift surrogate, by pair (rows: delta area, cols: spindle area)")
plt.tight_layout()
save_fig(fig, "03_surrogate")
plt.show()

### Surrogate for the unique-spindle count

Null for Yixiao's one-event rule (fraction of spindles coupled).

In [ ]:
from modules.event_coupling.coupling_unique import count_sequences_unique


def _one_unique_iter(
    first_df, s, first_wave, second_wave, sp, rng, min_shift_ms, scope, window_ms
):
    """One circular-shift iteration for the unique-count null (mirrors the
    loop body inside surrogate.surrogate_unique_count)."""
    parts = []
    if scope == "recording":
        for _, grp in s.groupby(sg.REC_KEYS):
            sh = sg._shift_within_recording(grp, sp, rng, min_shift_ms)
            if sh is not None:
                parts.append(sh)
    else:
        spans = sg._bout_spans(s, sp)
        for key, grp in s.groupby(sg.BOUT_KEYS):
            if key not in spans.index:
                continue
            lo, hi = spans.loc[key, "lo"], spans.loc[key, "hi"]
            pk = sg.circular_shift_peaks(grp[sp].to_numpy(), lo, hi, rng, min_shift_ms)
            if pk is None:
                continue
            sh = grp.copy()
            sh[sp] = pk
            parts.append(sh)

    if not parts:
        return np.nan
    shifted = pd.concat(parts, ignore_index=True)
    return count_sequences_unique(
        first_df, shifted, first_wave, second_wave, window_ms=window_ms
    )["n_sequences"].sum()


def surrogate_unique_count_ckpt(
    first_df,
    second_df,
    first_wave,
    second_wave,
    window_ms=None,
    n_iter=1000,
    seed=0,
    min_shift_ms=1000,
    scope="recording",
    checkpoint_every=None,
    checkpoint_fn=None,
    resume_from=None,
    verbose=True,
):
    """surrogate.surrogate_unique_count, with checkpoint/resume added."""
    obs = count_sequences_unique(
        first_df, second_df, first_wave, second_wave, window_ms=window_ms
    )
    real = int(obs["n_sequences"].sum())
    n_second = int(obs[f"n_{second_wave}"].sum())

    sp = f"{second_wave}_peak_index"
    s = second_df.dropna(subset=sg.BOUT_KEYS + [sp])

    rng = np.random.default_rng(seed)
    null = np.full(n_iter, np.nan)
    start = 0

    if resume_from is not None:
        prev_n = min(int(resume_from.get("n_done", 0)), n_iter)
        null[:prev_n] = resume_from["null"][:prev_n]
        start = prev_n
        if verbose:
            print(f"    resuming from {start}/{n_iter} iterations")
        for _ in range(start):  # advance the RNG to match a from-scratch run
            _one_unique_iter(
                first_df,
                s,
                first_wave,
                second_wave,
                sp,
                rng,
                min_shift_ms,
                scope,
                window_ms,
            )

    for i in range(start, n_iter):
        null[i] = _one_unique_iter(
            first_df,
            s,
            first_wave,
            second_wave,
            sp,
            rng,
            min_shift_ms,
            scope,
            window_ms,
        )
        done = i + 1
        if verbose and done % 100 == 0:
            print(f"    {done}/{n_iter}")
        if checkpoint_every and checkpoint_fn and done % checkpoint_every == 0:
            checkpoint_fn(
                {
                    "null": null.copy(),
                    "real": real,
                    "n_second": n_second,
                    "n_done": done,
                }
            )

    mu, sdv = np.nanmean(null), np.nanstd(null)
    p = (1 + np.nansum(null >= real)) / (1 + np.sum(~np.isnan(null)))
    return {
        "real": real,
        "null": null,
        "mu": float(mu),
        "sd": float(sdv),
        "z": float((real - mu) / sdv) if sdv > 0 else np.nan,
        "p": float(p),
        "ratio": float(real / mu) if mu > 0 else np.nan,
        "n_spindle": n_second,
    }

In [ ]:
res_unique = {}
for d_area, s_area in PAIRS:
    pair_label = f"{d_area} delta - {s_area} spindle"
    ckpt_path = CKPT_DIR / f"step13b_{d_area}_{s_area}.pkl"
    resume = None
    if ckpt_path.exists():
        with open(ckpt_path, "rb") as f:
            resume = pickle.load(f)

    if resume is not None and resume["n_done"] >= 1000:
        print(
            f"{pair_label}: already complete at {resume['n_done']}/1000 -- using checkpoint directly"
        )
        mu, sdv = np.nanmean(resume["null"]), np.nanstd(resume["null"])
        real = resume["real"]
        u = {
            "real": real,
            "null": resume["null"],
            "mu": float(mu),
            "sd": float(sdv),
            "z": float((real - mu) / sdv) if sdv > 0 else np.nan,
            "p": float(
                (1 + np.nansum(resume["null"] >= real))
                / (1 + np.sum(~np.isnan(resume["null"])))
            ),
            "ratio": float(real / mu) if mu > 0 else np.nan,
            "n_spindle": resume["n_second"],
        }
        res_unique[(d_area, s_area)] = u
        print(
            f"{pair_label:<22}: observed {u['real']:>7,} of {u['n_spindle']:>7,} spindles "
            f"({100 * u['real'] / u['n_spindle']:>5.1f}%) | null {u['mu']:>8,.0f} "
            f"| ratio {u['ratio']:.3f} | z {u['z']:+6.1f}"
        )
        continue

    if resume is not None:
        print(f"{pair_label}: resuming from {resume['n_done']}/1000 iterations")

    u = surrogate_unique_count_ckpt(
        dlt_s[dlt_s["area"] == d_area],
        spn_s[spn_s["area"] == s_area],
        "delta",
        "spindle",
        window_ms=WINDOW_MS,
        n_iter=1000,
        seed=0,
        scope="recording",
        verbose=False,
        checkpoint_every=100,
        checkpoint_fn=make_checkpoint_fn(f"step13b_{d_area}_{s_area}"),
        resume_from=resume,
    )
    res_unique[(d_area, s_area)] = u
    print(
        f"{pair_label:<22}: observed {u['real']:>7,} of {u['n_spindle']:>7,} spindles "
        f"({100 * u['real'] / u['n_spindle']:>5.1f}%) | null {u['mu']:>8,.0f} "
        f"| ratio {u['ratio']:.3f} | z {u['z']:+6.1f}"
    )

### Checks

In [ ]:
for key, r in res_surr.items():
    lo = left < -BASELINE_MS
    hi = left >= BASELINE_MS
    print(
        f"{key}: left tail z {np.nanmean(r['z'][lo]):+.2f} | "
        f"right tail z {np.nanmean(r['z'][hi]):+.2f}"
    )

## Delta phase (Hilbert transform), not run yet

Are spindles locked to the phase of the ongoing delta rhythm?

Continuous LFP band-pass filtered, Hilbert transform for phase, all nine pairs.

Delta band 0.5-4 Hz (confirmed with Yixiao). The LFP is read one trial file at a time from `PreprocessedData` (peak indices are within-trial), on the channel the delta waves were detected on.

In [ ]:
import io
import shutil
import time as _time

import scipy.io
from scipy.signal import butter, hilbert, sosfiltfilt

FS_HZ = 1000.0  # 1 sample = 1 ms
DELTA_BAND = (0.5, 4.0)  # Hz (confirmed with Yixiao)

# ROOT / Rat_HM_Ephys_TD_Analysis_R1_8-or-R9_16 / <rat-group> / PreprocessedData /
#   <area> / <rat> / <date> / <phase> / chan<channel>_<trial>.mat   (unsplit days: chan<channel>.mat)
ANALYSIS_GROUPS = {
    "Rat_HM_Ephys_TD_Analysis_R1_8": ["R1-4", "R5-8"],
    "Rat_HM_Ephys_TD_Analysis_R9_16": ["R9-12", "R13-16"],
}


def _find_preproc_rat_dir(area, rat):
    """PreprocessedData/<area>/<rat>, trying every ANALYSIS_GROUPS folder."""
    for analysis, subgroups in ANALYSIS_GROUPS.items():
        for sub in subgroups:
            cand = ROOT / analysis / sub / "PreprocessedData" / area / str(rat)
            if cand.exists():
                return cand
    return None


def channel_for(delta_df, rat, area, date):
    """Recording channel of this rat/area/date's delta detection (from the event table)."""
    rows = delta_df[
        (delta_df["rat"] == rat)
        & (delta_df["area"] == area)
        & (delta_df["date"] == date)
    ]
    chans = rows["channel"].dropna().unique()
    if len(chans) == 0:
        return None
    if len(chans) > 1:
        print(
            f"    warning: rat {rat} {area} {pd.Timestamp(date).date()} has "
            f"several channels {list(chans)}, using {chans[0]}"
        )
    return int(chans[0])


class UnreadableMatFile(FileNotFoundError):
    """The .mat file exists but could not be read (network / drive problem)."""


unreadable_lfp = []  # files skipped after all retries


def _load_mat_signal(
    path, chunk_bytes=8 * 1024 * 1024, attempts=6, waits=(10, 20, 30, 40, 60)
):
    """'data' array of one .mat file. Reading straight off Z: with loadmat(path)
    failed with OSError [Errno 22] part-way through a file, so the file is copied
    into memory in 8 MB pieces first, and a failed read is retried."""
    last = None
    for attempt in range(1, attempts + 1):
        try:
            buf = io.BytesIO()
            with open(path, "rb") as f:
                shutil.copyfileobj(f, buf, chunk_bytes)
            buf.seek(0)
            return scipy.io.loadmat(buf)["data"].squeeze()
        except OSError as e:
            last = e
            if attempt < attempts:
                _time.sleep(waits[min(attempt - 1, len(waits) - 1)])
    raise UnreadableMatFile(f"{path} unreadable after {attempts} attempts: {last}")


def load_trial_lfp(rat, date, trial, area, channel, phase="postsleep"):
    """LFP of one trial ('data' in chan<channel>_<trial>.mat, or chan<channel>.mat
    for unsplit days). None if the file isn't there."""
    rat_dir = _find_preproc_rat_dir(area, rat)
    if rat_dir is None:
        return None
    phase_dir = rat_dir / pd.to_datetime(date).strftime("%Y%m%d") / phase
    for f in (
        phase_dir / f"chan{channel}_{int(trial)}.mat",
        phase_dir / f"chan{channel}.mat",
    ):
        if f.exists():
            try:
                return _load_mat_signal(f).ravel()
            except UnreadableMatFile as e:
                unreadable_lfp.append(str(e))
                return None
    return None


def bandpass_delta(signal, fs=FS_HZ, band=DELTA_BAND, order=4):
    """Zero-phase Butterworth band-pass into the delta band. Second-order sections:
    the (b, a) form of this filter is unstable at 1000 Hz (output blows up)."""
    sos = butter(order, band, btype="band", fs=fs, output="sos")
    return sosfiltfilt(sos, signal)


def delta_phase_at_times(signal, fs, event_sample_indices):
    """Delta phase (radians, -pi..pi) at each sample index; NaN outside the trace."""
    phase = np.angle(hilbert(bandpass_delta(signal, fs=fs)))
    out = np.full(len(event_sample_indices), np.nan)
    ok = (event_sample_indices >= 0) & (event_sample_indices < len(phase))
    out[ok] = phase[event_sample_indices[ok]]
    return out


def event_phases(ev_df, peak_col, area, delta_df):
    """Delta phase in `area` at every event peak, one trial file at a time
    (peak indices are within-trial sample indices)."""
    out = []
    for (rat, date, trial), g in ev_df.groupby(["rat", "date", "trial"]):
        channel = channel_for(delta_df, rat, area, date)
        if channel is None:
            continue
        sig = load_trial_lfp(rat, date, trial, area, channel)
        if sig is None:
            continue
        peaks = g[peak_col].dropna().to_numpy().astype(int)
        if len(peaks):
            out.append(delta_phase_at_times(sig, FS_HZ, peaks))
    ph = np.concatenate(out) if out else np.array([])
    return ph[~np.isnan(ph)]


def event_phase_diffs(ev_df, peak_col, area_a, area_b, delta_df):
    """Delta phase difference area_a - area_b (wrapped to -pi..pi) at every event peak."""
    out = []
    for (rat, date, trial), g in ev_df.groupby(["rat", "date", "trial"]):
        ch_a = channel_for(delta_df, rat, area_a, date)
        ch_b = channel_for(delta_df, rat, area_b, date)
        if ch_a is None or ch_b is None:
            continue
        sig_a = load_trial_lfp(rat, date, trial, area_a, ch_a)
        sig_b = load_trial_lfp(rat, date, trial, area_b, ch_b)
        if sig_a is None or sig_b is None:
            continue
        peaks = g[peak_col].dropna().to_numpy().astype(int)
        if len(peaks):
            d = delta_phase_at_times(sig_a, FS_HZ, peaks) - delta_phase_at_times(
                sig_b, FS_HZ, peaks
            )
            out.append((d[~np.isnan(d)] + np.pi) % (2 * np.pi) - np.pi)
    return np.concatenate(out) if out else np.array([])


def rayleigh_test(phases):
    """Rayleigh test for non-uniformity of a circular distribution.

    Returns (R, z, p). Standard large-sample approximation
    (Zar, Biostatistical Analysis). Valid for n > ~10; for smaller samples
    treat the p-value as approximate.
    """
    n = len(phases)
    if n == 0:
        return np.nan, np.nan, np.nan
    C = np.sum(np.cos(phases))
    S = np.sum(np.sin(phases))
    R = np.sqrt(C**2 + S**2) / n
    z = n * R**2
    p = np.exp(-z) * (
        1
        + (2 * z - z**2) / (4 * n)
        - (24 * z - 132 * z**2 + 76 * z**3 - 9 * z**4) / (288 * n**2)
    )
    return R, z, max(min(p, 1.0), 0.0)


def circular_hist(ax, phases, bins=24, color="tab:purple"):
    """Phase histogram on a polar axis."""
    counts, edges = np.histogram(phases, bins=bins, range=(-np.pi, np.pi))
    width = edges[1] - edges[0]
    ax.bar(
        edges[:-1],
        counts,
        width=width,
        align="edge",
        color=color,
        alpha=0.75,
        edgecolor="white",
    )
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)

In [ ]:
# d_area delta phase at s_area spindle peaks
phase_by_pair = {}
for d_area, s_area in PAIRS:
    phase_by_pair[(d_area, s_area)] = event_phases(
        spn_ctx[spn_ctx["area"] == s_area], "spindle_peak_index", d_area, dlt_ctx
    )
for key, phases in phase_by_pair.items():
    print(f"{key}: {len(phases)} spindle-phase samples collected")

In [ ]:
# rows = delta area, columns = spindle area
fig, axes = plt.subplots(3, 3, figsize=(13, 12), subplot_kw={"projection": "polar"})
for (d_area, s_area), phases in phase_by_pair.items():
    ax = axes[AREAS.index(d_area), AREAS.index(s_area)]
    circular_hist(ax, phases)
    if len(phases):
        R, z, p = rayleigh_test(phases)
        ax.set_title(f"{d_area}-{s_area}\nR={R:.3f}, p={p:.4f}", fontsize=9)
    else:
        ax.set_title(f"{d_area}-{s_area}\n(no data yet)", fontsize=9)
fig.suptitle(
    "Delta phase at spindle peak times, by pair (rows: delta area, cols: spindle area)",
    fontsize=12,
)
plt.tight_layout()
save_fig(fig, "18_delta_phase_per_pair")
plt.show()

### Cross-area phase relationship

For each spindle area: do two delta areas keep a consistent phase difference at the spindle peaks?

In [ ]:
from itertools import combinations

cross_area_diffs = {}
for s_area in AREAS:
    s_area_df = spn_ctx[spn_ctx["area"] == s_area]
    for area_a, area_b in combinations(AREAS, 2):
        cross_area_diffs[(s_area, area_a, area_b)] = event_phase_diffs(
            s_area_df, "spindle_peak_index", area_a, area_b, dlt_ctx
        )

if unreadable_lfp:
    print(f"{len(unreadable_lfp)} LFP files could not be read and were skipped:")
    for msg in unreadable_lfp:
        print("   ", msg)

fig, axes = plt.subplots(
    len(AREAS), 3, figsize=(12, 4 * len(AREAS)), subplot_kw={"projection": "polar"}
)
for i, s_area in enumerate(AREAS):
    for j, (area_a, area_b) in enumerate(combinations(AREAS, 2)):
        ax = axes[i, j]
        diffs = cross_area_diffs[(s_area, area_a, area_b)]
        circular_hist(ax, diffs, color="tab:teal")
        if len(diffs):
            R, z, p = rayleigh_test(diffs)
            ax.set_title(
                f"{area_a}-{area_b} @ {s_area} spindles\nR={R:.3f}, p={p:.4f}",
                fontsize=8,
            )
        else:
            ax.set_title(
                f"{area_a}-{area_b} @ {s_area} spindles\n(no data yet)", fontsize=8
            )
fig.suptitle(
    "Delta-delta phase difference between areas, at each spindle area's own peak times",
    fontsize=11,
)
plt.tight_layout()
save_fig(fig, "19_cross_area_delta_phase_diff")
plt.show()

---
# Does coupling change with learning?

## HC vs Training per rat

Per rat, paired. With n = 7 a sign test needs 7/7 for p = 0.008 (6/7 gives 0.06).

In [ ]:
MIN_NREM_MIN = 10  # minutes of NREM sleep required for a rat-day to count
# rat-days with < 10 min NREM excluded (only here, not for the histograms or surrogates)

day_nrem = bout_dur.groupby(["rat", "date"])["nrem_bout_duration_s"].sum() / 60
good_days = day_nrem[day_nrem >= MIN_NREM_MIN].reset_index()[["rat", "date"]]
print(
    f"rat-days: {day_nrem.shape[0]} total -> {len(good_days)} with >= {MIN_NREM_MIN} min "
    f"NREM ({day_nrem.shape[0] - len(good_days)} dropped)"
)


def ratio_per_rat(dlt_df, spn_df, d_area, s_area, split_col):
    """One D->S ratio per rat per level of split_col, for one pair.

    Only rat-days in good_days (>= MIN_NREM_MIN minutes of NREM) are used;
    nrem_days = number of those rat-days behind each ratio.
    """
    d = dlt_df[dlt_df["area"] == d_area].merge(good_days, on=["rat", "date"])
    s_all = spn_df[spn_df["area"] == s_area].merge(good_days, on=["rat", "date"])
    rows = []
    for (rat, level), dg in d.groupby(["rat", split_col], observed=True):
        sg = s_all[(s_all["rat"] == rat) & (s_all[split_col] == level)]
        if sg.empty or dg.empty:
            continue
        lg = lh.collect_lags(dg, sg, "delta", "spindle", max_lag_ms=MAX_LAG_MS)
        if len(lg) == 0:
            continue
        n_days = dg[["rat", "date"]].drop_duplicates().shape[0]
        res = lh.summarise(
            lg, EDGES, baseline_ms=BASELINE_MS, windows=WINDOWS, verbose=False
        )
        rows.append(
            {
                "rat": rat,
                split_col: level,
                "pair": f"{d_area}_{s_area}",
                "d_area": d_area,
                "s_area": s_area,
                "ratio": res["windows"]["D->S"]["ratio"],
                "peak_ms": res["peak_ms"],
                "pairs": len(lg),
                "nrem_days": n_days,
            }
        )
    return pd.DataFrame(rows)


def compare(table, split_col, levels, ref, pairs=None):
    """Paired comparison of each level against `ref`, per pair."""
    pairs = pairs or PAIRS
    for d_area, s_area in pairs:
        pair_label = f"{d_area} delta - {s_area} spindle"
        sub = table[table["pair"] == f"{d_area}_{s_area}"]
        w = sub.pivot(index="rat", columns=split_col, values="ratio").dropna()
        if len(w) < 3:
            print(f"{pair_label}: only {len(w)} rats -- skipping")
            continue
        n_days = sub.pivot(index="rat", columns=split_col, values="nrem_days")
        print(f"\n{pair_label}  (n={len(w)} rats)")
        print(
            "  NREM-qualifying days per rat, by level: "
            + " | ".join(
                f"{l} median {n_days[l].median():.0f}"
                for l in levels
                if l in n_days.columns
            )
        )
        print("  means: " + " | ".join(f"{l} {w[l].mean():.3f}" for l in levels))
        for l in [x for x in levels if x != ref]:
            d = w[l] - w[ref]
            print(
                f"  {l:>7} vs {ref}: {d.mean():+.3f} | "
                f"{int((d > 0).sum())}/{len(w)} rats higher"
            )


hc_tr = pd.concat(
    [
        ratio_per_rat(dlt_ctx, spn_ctx, d_area, s_area, "group")
        for d_area, s_area in PAIRS
    ],
    ignore_index=True,
)
hc_tr = hc_tr[hc_tr["group"].isin(["HC", "Training"])]

print("rats with enough qualifying days per cell:")
print(hc_tr.groupby(["pair", "group"])["rat"].nunique().to_string())

In [ ]:
compare(hc_tr, "group", ["HC", "Training"], ref="HC")

## Training split by session type

Training split into new goal location, probe and normal. Prediction: NGL highest.

In [ ]:
tr_type = pd.concat(
    [
        ratio_per_rat(dlt_ctx, spn_ctx, d_area, s_area, "session_type")
        for d_area, s_area in PAIRS
    ],
    ignore_index=True,
)

print(tr_type.groupby(["pair", "session_type"])["rat"].nunique().to_string())

In [ ]:
compare(tr_type, "session_type", ["NGL", "probe", "normal"], ref="normal")

## HC vs each training type

Sleep after new learning vs HC. Third version of the same question, so treat results as leads.

In [ ]:
def add_condition4(df):
    """HC and the three training types in one column."""
    df = df.copy()
    df["condition4"] = np.where(
        df["group"] == "HC", "HC", df["session_type"].astype(str)
    )
    df.loc[~df["condition4"].isin(["HC", "NGL", "probe", "normal"]), "condition4"] = (
        np.nan
    )
    return df


four = pd.concat(
    [
        ratio_per_rat(
            add_condition4(dlt_ctx),
            add_condition4(spn_ctx),
            d_area,
            s_area,
            "condition4",
        )
        for d_area, s_area in PAIRS
    ],
    ignore_index=True,
)

compare(four, "condition4", ["HC", "NGL", "probe", "normal"], ref="HC")

---
# Per-rat figures

## Histogram for every rat

Each curve divided by its own baseline (1.0 = chance).

In [ ]:
COHORTS = {
    "build-up (rats 1-8)": [r for r in RATS if r <= 8],
    "update (rats 9-16)": [r for r in RATS if r > 8],
}

dlt_q = dlt.merge(good_days, on=["rat", "date"])
spn_q = spn.merge(good_days, on=["rat", "date"])

for coh_label, coh_rats in COHORTS.items():
    # 3x3 grid: rows = delta area, columns = spindle area (AREAS order).
    fig, axes = plt.subplots(3, 3, figsize=(14, 11), sharex=True, sharey=True)
    for d_area, s_area in PAIRS:
        ax = axes[AREAS.index(d_area), AREAS.index(s_area)]
        for rat in coh_rats:
            dg = dlt_q[(dlt_q["area"] == d_area) & (dlt_q["rat"] == rat)]
            sg_ = spn_q[(spn_q["area"] == s_area) & (spn_q["rat"] == rat)]
            if dg.empty or sg_.empty:
                continue  # no qualifying NREM days for this rat
            lg = lh.collect_lags(dg, sg_, "delta", "spindle", max_lag_ms=MAX_LAG_MS)
            r = lh.summarise(
                lg, EDGES, baseline_ms=BASELINE_MS, windows=WINDOWS, verbose=False
            )
            ax.plot(left, r["counts"], lw=1.4, alpha=0.85, label=f"rat {rat}")
        ax.axvline(0, color="k", lw=0.8)
        ax.set_title(f"{d_area} delta vs {s_area} spindle", fontsize=10)
    for i, d_area in enumerate(AREAS):
        axes[i, 0].set_ylabel(f"count\n({d_area} delta)")
    for j, s_area in enumerate(AREAS):
        axes[-1, j].set_xlabel(f"lag (ms)\n({s_area} spindle)")
    axes[0, 0].legend(fontsize=8)
    fig.suptitle(coh_label)
    plt.tight_layout()
    tag = "buildup" if "1-8" in coh_label else "update"
    save_fig(fig, f"04_per_rat_{tag}")
    plt.show()

## Paired HC vs Training

One line per rat.

In [ ]:
def plot_paired(table, split_col, a, b, title, figname=None):
    # 3x3 grid: rows = delta area, columns = spindle area (AREAS order).
    fig, axes = plt.subplots(3, 3, figsize=(13, 11), sharey=False)
    for d_area, s_area in PAIRS:
        ax = axes[AREAS.index(d_area), AREAS.index(s_area)]
        pair_key = f"{d_area}_{s_area}"
        w = (
            table[table["pair"] == pair_key]
            .pivot(index="rat", columns=split_col, values="ratio")
            .dropna()
        )
        if w.empty or a not in w or b not in w:
            ax.set_title(f"{d_area}-{s_area}: no data", fontsize=9)
            continue
        for rat, row in w.iterrows():
            up = row[b] > row[a]
            ax.plot(
                [0, 1],
                [row[a], row[b]],
                marker="o",
                lw=1.2,
                alpha=0.8,
                color="tab:blue" if up else "tab:red",
            )
            ax.annotate(str(rat), (1.03, row[b]), fontsize=7, va="center")
        ax.axhline(1, color="k", lw=1, ls=":")
        ax.set_xticks([0, 1])
        ax.set_xticklabels([a, b])
        ax.set_xlim(-0.3, 1.4)
        ax.set_title(
            f"{d_area}-{s_area}  ({int((w[b] > w[a]).sum())}/{len(w)} higher in {b})",
            fontsize=9,
        )
    for i, d_area in enumerate(AREAS):
        axes[i, 0].set_ylabel(f"D->S ratio\n({d_area} delta)")
    fig.suptitle(title + "   -- blue: higher in " + b + ", red: lower")
    plt.tight_layout()
    if figname:
        save_fig(fig, figname)
    plt.show()


plot_paired(
    hc_tr,
    "group",
    "HC",
    "Training",
    "Home cage vs Training, per rat",
    figname="05_paired_hc_training",
)

In [ ]:
plot_paired(
    tr_type,
    "session_type",
    "normal",
    "NGL",
    "Normal training vs new goal location, per rat",
    figname="06_paired_ngl_normal",
)

## Does the effect size follow spindle rate?

Effect size vs spindle rate in the spindle area.

In [ ]:
nrem_min = bout_dur.groupby("rat")["nrem_bout_duration_s"].sum() / 60

# 3x3 grid: rows = delta area, columns = spindle area (AREAS order).
fig, axes = plt.subplots(3, 3, figsize=(13, 11), sharey=False)
for d_area, s_area in PAIRS:
    ax = axes[AREAS.index(d_area), AREAS.index(s_area)]
    rate = (spn[spn["area"] == s_area].groupby("rat").size() / nrem_min).rename(
        "spindle_rate"
    )
    pair_key = f"{d_area}_{s_area}"
    sub = (
        hc_tr[(hc_tr["pair"] == pair_key) & (hc_tr["group"] == "Training")]
        .set_index("rat")
        .join(rate)
    )
    ax.scatter(sub["spindle_rate"], sub["ratio"], s=45)
    for rat, row in sub.iterrows():
        ax.annotate(
            str(rat),
            (row["spindle_rate"], row["ratio"]),
            fontsize=8,
            xytext=(4, 3),
            textcoords="offset points",
        )
    ax.axhline(1, color="k", lw=1, ls=":")
    ax.set_xlabel(f"{s_area} spindle rate (events / min NREM)")
    ax.set_title(f"{d_area}-{s_area}", fontsize=9)
for i, d_area in enumerate(AREAS):
    axes[i, 0].set_ylabel(f"D->S ratio\n({d_area} delta)")
plt.tight_layout()
save_fig(fig, "07_ratio_vs_spindle_rate")
plt.show()

# Coupling over time

Per-rat Spearman + permutation on the D->S ratio (fraction of spindles with a delta 100-1300 ms before, `count_sequences_unique`).

## Per-day ratio table

Computed once per pair and used by all plots below. Rat-days with less than 10 min of NREM (`MIN_NREM_MIN`) are removed and listed in a footnote under each plot.

In [ ]:
from modules.event_coupling.coupling_unique import count_sequences_unique

# days since each rat's first recording (added here if missing)
if "days_elapsed" not in dlt_ctx.columns:
    first = dlt_ctx.groupby("rat")["date"].transform("min")
    dlt_ctx["days_elapsed"] = (dlt_ctx["date"] - first).dt.days


def per_day_sequence_ratio(dlt_df, spn_df, d_area, s_area, window_ms):
    """Per rat-day: fraction of spindles (in s_area) with at least one delta
    (in d_area) preceding them within window_ms -- Yixiao's one-event rule,
    applied day by day instead of pooled across the whole dataset. Mirrors
    the cross-regional notebooks' per_day_cooccurrence (n_coupled/n_seed),
    but using the sequential (lag-window) definition of coupling instead of
    the overlap-based nearest-neighbor one."""
    d_all = dlt_df[dlt_df["area"] == d_area]
    s_all = spn_df[spn_df["area"] == s_area]
    rows = []
    for (rat, date), sg in s_all.groupby(["rat", "date"]):
        dg = d_all[(d_all["rat"] == rat) & (d_all["date"] == date)]
        if dg.empty or sg.empty:
            continue
        u = count_sequences_unique(dg, sg, "delta", "spindle", window_ms=window_ms)
        n_coupled, n_seed = int(u["n_sequences"].sum()), int(u["n_spindle"].sum())
        if n_seed == 0:
            continue
        meta = dlt_df[(dlt_df["rat"] == rat) & (dlt_df["date"] == date)]
        if meta.empty:
            continue
        meta = meta.iloc[0]
        rows.append(
            {
                "rat": rat,
                "date": date,
                "n_coupled": n_coupled,
                "n_seed": n_seed,
                "group": meta["group"],
                "days_elapsed": meta["days_elapsed"],
                "gl_num": meta.get("gl_num"),
                "session_num": meta.get("session_num"),
            }
        )
    return pd.DataFrame(rows)


# NREM cutoff: rat-days with < MIN_NREM_MIN minutes of NREM are removed
# here and listed in the footnote of each over-time plot
_nrem = {(int(r), pd.Timestamp(d).normalize()): m for (r, d), m in day_nrem.items()}


def apply_nrem_cutoff(daily_df):
    """Split daily_df into rat-days with >= MIN_NREM_MIN minutes of NREM and the removed ones."""
    if daily_df.empty:
        return daily_df, daily_df.assign(nrem_min=[])
    mins = pd.Series(
        [
            _nrem.get((int(r), pd.Timestamp(d).normalize()), 0.0)
            for r, d in zip(daily_df["rat"], daily_df["date"])
        ],
        index=daily_df.index,
    )
    keep = mins >= MIN_NREM_MIN
    removed = daily_df.loc[~keep, ["rat", "date", "group"]].assign(
        nrem_min=mins[~keep].values
    )
    return daily_df[keep].reset_index(drop=True), removed.reset_index(drop=True)


def nrem_removed_keys(d_area, s_area):
    rem = nrem_removed.get((d_area, s_area))
    return set() if rem is None else set(zip(rem["rat"], rem["date"]))


def add_nrem_footnote(fig, d_area, s_area, groups):
    """List the rat-days removed by the NREM cutoff under the figure."""
    rem = nrem_removed.get((d_area, s_area))
    if rem is None or rem.empty:
        return
    rem = rem[rem["group"].isin(groups)]
    if rem.empty:
        return
    parts = []
    for rat, g in rem.groupby("rat"):
        days = ", ".join(
            f"{pd.Timestamp(d).date()} {grp} ({m:.1f} min)"
            for d, grp, m in zip(g["date"], g["group"], g["nrem_min"])
        )
        parts.append(f"Rat {rat}: {days}")
    note = f"Removed, < {MIN_NREM_MIN} min NREM: " + " | ".join(parts)
    old = [t for t in fig.texts if t.get_color() == "firebrick"]
    if old:
        old[0].set_text(old[0].get_text() + "\n" + note)
        fig.subplots_adjust(bottom=0.36)
    else:
        fig.subplots_adjust(bottom=0.30)
        fig.text(
            0.5,
            0.02,
            note,
            ha="center",
            va="bottom",
            fontsize=8,
            color="firebrick",
            wrap=True,
        )


nrem_removed = {}

daily_cache = {}
for d_area, s_area in PAIRS:
    daily_df = per_day_sequence_ratio(dlt_ctx, spn_ctx, d_area, s_area, WINDOW_MS)
    daily_df, nrem_removed[(d_area, s_area)] = apply_nrem_cutoff(daily_df)
    daily_cache[(d_area, s_area)] = daily_df
    print(
        f"{d_area} delta - {s_area} spindle: {len(daily_df)} rat-days with data, "
        f"{len(nrem_removed[(d_area, s_area)])} removed (< {MIN_NREM_MIN} min NREM)"
    )

## Training days over time

y = ratio. Training days only, split by cohort, x = GL_session with goal-location blocks shaded.

In [ ]:
from scipy.stats import spearmanr


def plot_training_overtime(daily_df, d_area, s_area):
    sub_all = daily_df.copy()
    sub_all["ratio_proxy"] = sub_all["n_coupled"] / sub_all["n_seed"].replace(0, np.nan)
    sub_all = sub_all.dropna(subset=["gl_num", "session_num", "ratio_proxy"])
    if sub_all.empty:
        print(
            f"{d_area} delta - {s_area} spindle: no GL/session-labelled data, skipping"
        )
        return []

    rng = np.random.default_rng(0)
    results = []
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.6), sharey=True)
    for ax, (label, rats) in zip(axes, COHORTS.items()):
        sub = sub_all[sub_all["rat"].isin(rats)].copy()
        if sub.empty:
            ax.set_title(label, fontsize=11)
            continue

        sub["step"] = sub["gl_num"] * 10 + sub["session_num"]
        order = sorted(sub["step"].unique())
        pos = {s: i for i, s in enumerate(order)}
        tick_labels = [f"GL{int(s // 10)}_S{int(s % 10)}" for s in order]

        for rat in rats:
            r = sub[sub["rat"] == rat].sort_values("days_elapsed")
            if len(r) < 5:
                continue
            rho, _ = spearmanr(r["days_elapsed"], r["ratio_proxy"])
            perm_rhos = np.array(
                [
                    spearmanr(r["days_elapsed"], rng.permutation(r["ratio_proxy"]))[0]
                    for _ in range(1000)
                ]
            )
            p_perm = (np.sum(np.abs(perm_rhos) >= abs(rho)) + 1) / 1001
            z_perm = (rho - perm_rhos.mean()) / (perm_rhos.std() + 1e-9)

            results.append(
                {
                    "pair": f"{d_area}_{s_area}",
                    "cohort": label,
                    "rat": rat,
                    "n_days": len(r),
                    "rho": rho,
                    "z_perm": z_perm,
                    "p_perm": p_perm,
                }
            )

            g = r.groupby("step")["ratio_proxy"].mean()

            xs = [pos[s] for s in g.index]
            ax.plot(
                xs,
                g.values,
                marker="o",
                ms=4,
                lw=1.2,
                alpha=0.8,
                label=f"rat {rat}  ρ={rho:+.2f}{'*' if p_perm < 0.05 else ''}",
            )

        for i, s in enumerate(order):
            if int(s // 10) % 2 == 0:
                ax.axvspan(i - 0.5, i + 0.5, color="0.92", zorder=0)
        ax.set_xticks(range(len(tick_labels)))
        ax.set_xticklabels(tick_labels, rotation=90, fontsize=7)
        ax.set_title(label, fontsize=11)
        ax.legend(fontsize=7, ncol=2)
        ax.grid(alpha=0.2, lw=0.5, axis="y")

    axes[0].set_ylabel("ratio (fraction of spindles coupled)")
    fig.suptitle(
        f"{d_area} delta - {s_area} spindle  --  D->S ratio over time, Training days   "
        "(shading marks goal-location blocks; * = permutation p < 0.05)",
        fontsize=12,
    )
    plt.tight_layout()
    add_nrem_footnote(fig, d_area, s_area, ["Training"])
    save_fig(fig, f"08_training_overtime_{d_area}_{s_area}")
    plt.show()
    return results


training_overtime_results = []
for d_area, s_area in PAIRS:
    daily_df = daily_cache[(d_area, s_area)]
    if daily_df.empty:
        print(f"{d_area}-{s_area}: no data, skipping")
        continue
    training_overtime_results.extend(plot_training_overtime(daily_df, d_area, s_area))

In [ ]:
training_overtime_summary = pd.DataFrame(training_overtime_results)
training_overtime_summary["significant"] = training_overtime_summary["p_perm"] < 0.05
print(f"{len(training_overtime_summary)} rat-level permutation tests total\n")
print(
    training_overtime_summary.sort_values(["pair", "p_perm"])[
        ["pair", "cohort", "rat", "n_days", "rho", "z_perm", "p_perm", "significant"]
    ].to_string(index=False)
)

## HC days over time

Does coupling drift on HC days? x = days_elapsed.

In [ ]:
def plot_hc_overtime(daily_df, d_area, s_area):
    sub_all = daily_df.copy()
    sub_all["ratio_proxy"] = sub_all["n_coupled"] / sub_all["n_seed"].replace(0, np.nan)
    sub_all = sub_all[sub_all["group"] == "HC"].dropna(subset=["ratio_proxy"])
    if sub_all.empty:
        print(f"{d_area} delta - {s_area} spindle: no HC data, skipping")
        return []

    rng = np.random.default_rng(0)
    results = []
    missing_notes = []
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.6), sharey=True)
    for ax, (label, rats) in zip(axes, COHORTS.items()):
        sub = sub_all[sub_all["rat"].isin(rats)].copy()
        if sub.empty:
            ax.set_title(label, fontsize=11)
            continue

        for rat in rats:
            r = sub[sub["rat"] == rat].sort_values("days_elapsed")
            if len(r) < 5:
                if len(r) > 0:
                    missing_notes.append(
                        f"Rat {rat}: only {len(r)} usable HC day(s) (need >= 5)"
                    )
                continue
            rho, _ = spearmanr(r["days_elapsed"], r["ratio_proxy"])
            perm_rhos = np.array(
                [
                    spearmanr(r["days_elapsed"], rng.permutation(r["ratio_proxy"]))[0]
                    for _ in range(1000)
                ]
            )
            p_perm = (np.sum(np.abs(perm_rhos) >= abs(rho)) + 1) / 1001
            z_perm = (rho - perm_rhos.mean()) / (perm_rhos.std() + 1e-9)

            results.append(
                {
                    "pair": f"{d_area}_{s_area}",
                    "cohort": label,
                    "rat": rat,
                    "n_days": len(r),
                    "rho": rho,
                    "z_perm": z_perm,
                    "p_perm": p_perm,
                }
            )

            ax.plot(
                r["days_elapsed"],
                r["ratio_proxy"],
                marker="o",
                ms=4,
                lw=1.2,
                alpha=0.8,
                label=f"rat {rat}  \u03c1={rho:+.2f}{'*' if p_perm < 0.05 else ''}",
            )

        ax.set_title(label, fontsize=11)
        ax.set_xlabel("days elapsed")
        ax.legend(fontsize=7, ncol=2)
        ax.grid(alpha=0.2, lw=0.5, axis="y")

    axes[0].set_ylabel("ratio (fraction of spindles coupled)")
    fig.suptitle(
        f"{d_area} delta - {s_area} spindle  --  D->S ratio over time, Home-cage days",
        fontsize=12,
    )
    plt.tight_layout()

    if missing_notes:
        fig.subplots_adjust(bottom=0.26)
        footnote = "Missing from plot: " + " | ".join(missing_notes)
        fig.text(
            0.5,
            0.02,
            footnote,
            ha="center",
            va="bottom",
            fontsize=8,
            color="firebrick",
            wrap=True,
        )

    add_nrem_footnote(fig, d_area, s_area, ["HC"])
    save_fig(fig, f"09_hc_overtime_{d_area}_{s_area}")
    plt.show()
    return results


hc_overtime_results = []
for d_area, s_area in PAIRS:
    daily_df = daily_cache[(d_area, s_area)]
    if daily_df.empty:
        print(f"{d_area}-{s_area}: no data, skipping")
        continue
    hc_overtime_results.extend(plot_hc_overtime(daily_df, d_area, s_area))

In [ ]:
hc_overtime_summary = pd.DataFrame(hc_overtime_results)
hc_overtime_summary["significant"] = hc_overtime_summary["p_perm"] < 0.05
print(f"{len(hc_overtime_summary)} rat-level permutation tests total\n")
print(
    hc_overtime_summary.sort_values(["pair", "p_perm"])[
        ["pair", "cohort", "rat", "n_days", "rho", "z_perm", "p_perm", "significant"]
    ].to_string(index=False)
)

## Normalized to the nearest HC session

Training ratio / ratio of the rat's nearest HC day (second-nearest if that one is 0). Missing or dropped points are listed in a footnote under each figure.

In [ ]:
def find_missing_points(daily_df, d_area, s_area, rats_to_check):
    """Training days with a valid GL/session label that are absent from
    daily_df entirely -- per_day_sequence_ratio's groupby skips any
    (rat, date) with zero d_area deltas or zero s_area spindles that day,
    rather than recording an explicit zero row, so comparing against
    dlt_ctx is the only way to catch this."""
    expected = dlt_ctx[
        (dlt_ctx["rat"].isin(rats_to_check))
        & (dlt_ctx["area"] == d_area)
        & (dlt_ctx["group"] == "Training")
    ]
    expected = expected.dropna(subset=["gl_num", "session_num"])[
        ["rat", "date", "gl_num", "session_num"]
    ].drop_duplicates()
    present = set(zip(daily_df["rat"], daily_df["date"])) | nrem_removed_keys(
        d_area, s_area
    )
    notes = []
    for _, row in expected.iterrows():
        if (row["rat"], row["date"]) not in present:
            date_str = (
                row["date"].date() if hasattr(row["date"], "date") else row["date"]
            )
            notes.append(
                f"Rat {row['rat']}: {date_str} (GL{int(row['gl_num'])}_S{int(row['session_num'])}) "
                f"missing -- no {d_area} delta or {s_area} spindle events that day"
            )
    return notes


def dropped_to_notes(dropped):
    notes = []
    for d in dropped:
        date_str = d["date"].date() if hasattr(d["date"], "date") else d["date"]
        notes.append(
            f"Rat {d['rat']}: {date_str} missing -- nearest two HC days "
            f"({d['nearest_hc_days_elapsed']}) both had 0 coupling"
        )
    return notes


def nearest_hc_normalize(daily_df):
    """For each training day, divide by the nearest home-cage day's
    ratio_proxy. If the nearest HC day is 0, falls back to the
    second-nearest. If that's also 0, the training day is dropped
    (capped at two attempts)."""
    daily_df = daily_df.copy()
    daily_df["ratio_proxy"] = daily_df["n_coupled"] / daily_df["n_seed"].replace(
        0, np.nan
    )
    out_rows = []
    dropped = []
    for rat, g in daily_df.groupby("rat"):
        hc = (
            g[g["group"] == "HC"]
            .dropna(subset=["ratio_proxy"])
            .sort_values("days_elapsed")
        )
        train = g[g["group"] == "Training"].dropna(
            subset=["ratio_proxy", "gl_num", "session_num"]
        )
        if hc.empty or train.empty:
            continue
        for _, row in train.iterrows():
            hc_sorted = hc.iloc[
                (hc["days_elapsed"] - row["days_elapsed"]).abs().argsort()
            ]
            used = None
            for rank, (_, hc_row) in enumerate(hc_sorted.iterrows()):
                if rank >= 2:
                    break
                if hc_row["ratio_proxy"] and hc_row["ratio_proxy"] > 0:
                    used = hc_row
                    break
            if used is None:
                nearest_two = hc_sorted.head(2)
                dropped.append(
                    {
                        "rat": rat,
                        "date": row["date"],
                        "days_elapsed": row["days_elapsed"],
                        "nearest_hc_days_elapsed": list(nearest_two["days_elapsed"]),
                    }
                )
                continue
            out_rows.append(
                {
                    **row.to_dict(),
                    "ratio": row["ratio_proxy"] / used["ratio_proxy"],
                    "nearest_hc_days_elapsed": used["days_elapsed"],
                }
            )
    return pd.DataFrame(out_rows), dropped


def plot_hc_normalized(
    norm_df, daily_df, missing_notes, d_area, s_area, norm_label, fname_tag
):
    """Shared by Steps 21 and 22 -- same GL-shaded, per-cohort layout as the
    raw over-time plot, with a chance line at 1.0 since this measure is
    meant to be read relative to that rat's own HC baseline. Footnotes any
    point missing entirely or dropped during normalization."""
    if norm_df.empty:
        print(f"{d_area} delta - {s_area} spindle: no data to normalize, skipping")
        return

    fig, axes = plt.subplots(1, 2, figsize=(15, 4.6), sharey=True)
    for ax, (label, rats) in zip(axes, COHORTS.items()):
        sub = norm_df[norm_df["rat"].isin(rats)].copy()
        if sub.empty:
            ax.set_title(label, fontsize=11)
            continue
        sub["step"] = sub["gl_num"] * 10 + sub["session_num"]
        order = sorted(sub["step"].unique())
        pos = {s: i for i, s in enumerate(order)}
        tick_labels = [f"GL{int(s // 10)}_S{int(s % 10)}" for s in order]

        for rat in rats:
            r = sub[sub["rat"] == rat]
            if len(r) < 3:
                continue
            g = r.groupby("step")["ratio"].mean()
            xs = [pos[s] for s in g.index]
            ax.plot(
                xs, g.values, lw=1.6, alpha=0.85, marker="o", ms=4, label=f"rat {rat}"
            )

        for i, s in enumerate(order):
            if int(s // 10) % 2 == 0:
                ax.axvspan(i - 0.5, i + 0.5, color="0.92", zorder=0)
        ax.axhline(1, color="k", lw=1.2, ls=":")
        ax.set_xticks(range(len(tick_labels)))
        ax.set_xticklabels(tick_labels, rotation=90, fontsize=7)
        ax.set_title(label, fontsize=11)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.2, lw=0.5, axis="y")

    axes[0].set_ylabel("ratio")
    fig.suptitle(
        f"{d_area} delta - {s_area} spindle  --  normalized to {norm_label}   "
        "(shading marks goal-location blocks)",
        fontsize=12,
    )
    plt.tight_layout()

    if missing_notes:
        fig.subplots_adjust(bottom=0.28)
        footnote = "Missing point(s): " + " | ".join(missing_notes)
        fig.text(
            0.5,
            0.02,
            footnote,
            ha="center",
            va="bottom",
            fontsize=8,
            color="firebrick",
            wrap=True,
        )

    add_nrem_footnote(fig, d_area, s_area, ["Training", "HC"])
    save_fig(fig, f"{fname_tag}_{d_area}_{s_area}")
    plt.show()


nearest_hcnorm_results = {}
all_dropped = []
for d_area, s_area in PAIRS:
    daily_df = daily_cache[(d_area, s_area)]
    if daily_df.empty:
        continue
    norm_df, dropped = nearest_hc_normalize(daily_df)
    nearest_hcnorm_results[(d_area, s_area)] = norm_df
    all_dropped.append((f"{d_area}-{s_area}", dropped))
    all_rats_in_cohorts = [r for rats in COHORTS.values() for r in rats]
    missing_notes = find_missing_points(
        daily_df, d_area, s_area, all_rats_in_cohorts
    ) + dropped_to_notes(dropped)
    plot_hc_normalized(
        norm_df,
        daily_df,
        missing_notes,
        d_area,
        s_area,
        "nearest home-cage session",
        "10_nearesthcnorm",
    )

print()
print(
    "=== Points removed (nearest-HC): both nearest and second-nearest HC days were 0 ==="
)
any_dropped = False
for name, dropped in all_dropped:
    for d in dropped:
        any_dropped = True
        print(
            f"  [{name}] rat {d['rat']}, {d['date']} (days_elapsed={d['days_elapsed']}): "
            f"nearest two HC days ({d['nearest_hc_days_elapsed']}) both had 0 coupling"
        )
if not any_dropped:
    print("  none -- every training day found a usable HC baseline within two tries")

## Normalized to the average HC session

Training ratio / average of the rat's nonzero HC days. Rats without a usable HC day and missing days are listed in the footnote.

In [ ]:
def average_hc_normalize(daily_df):
    daily_df = daily_df.copy()
    daily_df["ratio_proxy"] = daily_df["n_coupled"] / daily_df["n_seed"].replace(
        0, np.nan
    )
    out_rows = []
    skipped = []
    for rat, g in daily_df.groupby("rat"):
        hc = g[g["group"] == "HC"].dropna(subset=["ratio_proxy"])
        hc_nonzero = hc[hc["ratio_proxy"] > 0]
        train = g[g["group"] == "Training"].dropna(
            subset=["ratio_proxy", "gl_num", "session_num"]
        )
        if train.empty:
            continue
        if hc_nonzero.empty:
            skipped.append({"rat": rat, "n_hc_days": len(hc)})
            continue
        avg_hc = hc_nonzero["ratio_proxy"].mean()
        for _, row in train.iterrows():
            out_rows.append(
                {
                    **row.to_dict(),
                    "ratio": row["ratio_proxy"] / avg_hc,
                    "avg_hc_value": avg_hc,
                    "n_hc_days_used": len(hc_nonzero),
                }
            )
    return pd.DataFrame(out_rows), skipped


avg_hcnorm_results = {}
all_skipped = []
for d_area, s_area in PAIRS:
    daily_df = daily_cache[(d_area, s_area)]
    if daily_df.empty:
        continue
    norm_df, skipped = average_hc_normalize(daily_df)
    avg_hcnorm_results[(d_area, s_area)] = norm_df
    if skipped:
        all_skipped.append((f"{d_area}-{s_area}", skipped))
    all_rats_in_cohorts = [r for rats in COHORTS.values() for r in rats]
    missing_notes = find_missing_points(daily_df, d_area, s_area, all_rats_in_cohorts)
    missing_notes += [
        f"Rat {sk['rat']}: no usable HC day at all ({sk['n_hc_days']} HC days, all 0)"
        for sk in skipped
    ]
    plot_hc_normalized(
        norm_df,
        daily_df,
        missing_notes,
        d_area,
        s_area,
        "average home-cage session",
        "11_avghcnorm",
    )

print()
if all_skipped:
    print("=== Rats with no usable HC baseline (every HC day was 0) ===")
    for name, rats in all_skipped:
        print(f"  [{name}] rats skipped: {[r['rat'] for r in rats]}")
else:
    print("Every rat had at least one usable HC day, for every pair.")

---
# Notes

## Still open

- Surrogates run on the 2,000-bout sample, not on all bouts.
- Pre-sleep not used.
- 4 h of post-sleep pooled (see the first-hour notebook).
- PL-PL, RSC-RSC, HPC-HPC share an electrode.

## For Yixiao

- Should the same-electrode pairs be treated as checks only?
- n = 7 (4 build-up, 3 update), so cohorts are pooled.